# 00 · Ejecutar el proyecto en Google Colab

**Proyecto 22 · Detector de noticias falsas** — Marco Andrés Rico

Este cuaderno reproduce el proyecto en un entorno limpio: instala dependencias, descarga y verifica los datos,
ejecuta las pruebas automáticas, muestra los resultados registrados y usa el detector.

**Opción A:** clonar desde GitHub (cambie la URL). **Opción B:** subir el `.zip` del repositorio con el panel
de archivos de Colab y descomprimirlo (celda alternativa).

In [ ]:
REPO_URL = "https://github.com/USUARIO/detector-noticias-falsas.git"  # <-- cambie por la URL del repositorio
!git clone -q {REPO_URL} detector-noticias-falsas
%cd detector-noticias-falsas

In [ ]:
# Opción B (en lugar de la celda anterior): suba detector-noticias-falsas.zip y ejecute
# !unzip -q detector-noticias-falsas.zip && %cd detector-noticias-falsas

## 1. Instalar dependencias (≈ 1 min)

In [ ]:
!pip install -q -r requirements.txt && pip install -q -e .
import sys, os; sys.path.insert(0, os.path.abspath("src"))

## 2. Descargar y preparar los datos
Se descargan desde un commit fijado del repositorio oficial y se verifica el SHA‑256.

In [ ]:
!python -m fakenews.data preparar

## 3. Pruebas automáticas

In [ ]:
!python -m pytest -q

## 4. Resultados registrados (bitácora y evaluación final)

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
bit = pd.read_csv("results/bitacora.csv")
display(bit[["id", "esquema", "cambio", "f1_macro_media", "f1_macro_de", "brecha_sobreajuste"]])
fin = json.load(open("results/metricas_finales.json", encoding="utf-8"))
print("Modelo elegido:", fin["seleccion"])
for k in ("prueba_dd", "prueba_fd"):
    m = fin[k]["modelo_final"]["metricas"]; ic = fin[k]["modelo_final"]["ic95"]["f1_macro"]
    print(f"{k}: F1-macro {m['f1_macro']:.3f} (IC95 {ic['ic95_inf']:.3f}-{ic['ic95_sup']:.3f}), "
          f"especificidad {m['especificidad']:.3f}, recall falsas {m['recall_falsa']:.3f}")
print(json.dumps(fin["criterios_exito"], indent=2))
for f in ["comparacion_experimentos", "curva_aprendizaje", "matrices_confusion", "subgrupos_prueba_fd"]:
    display(Image(f"results/figuras/{f}.png"))

## 5. Usar el detector

In [ ]:
from fakenews.predict import Detector, EntradaInvalida
det = Detector()
for aviso in det.avisos_carga: print("AVISO:", aviso)   # si aparece, ejecute: !python -m fakenews.train
titular = "Alcaldía anuncia cierre parcial de una vía principal por obras"
cuerpo = ("La Secretaría de Movilidad informó el lunes que la vía principal tendrá un cierre parcial desde el 2 de "
          "marzo. Según el comunicado, durará tres semanas y habrá desvíos señalizados. \"Pedimos planear los viajes\", "
          "dijo la secretaria en rueda de prensa. La entidad explicó que los trabajos serán nocturnos.")
r = det.predecir(titular, cuerpo)
print(r["etiqueta"], r["prob_falsa"]); print(r["reglas"]["explicacion"])
pd.DataFrame(r["terminos"]["hacia_falsa"] + r["terminos"]["hacia_verdadera"])

In [ ]:
try:
    det.predecir("", "Texto muy corto")
except EntradaInvalida as e:
    print("Entrada rechazada con mensaje claro:", e)

## 6. (Opcional) Reproducir todo desde cero
La bitácora completa tarda ≈ 1,5 h en CPU. La evaluación final ya se usó una vez; para reproducirla se añade
`--forzar` y la repetición queda anotada en `results/uso_prueba.log`.

In [ ]:
# !python -m fakenews.experiments
# !python -m fakenews.evaluate --confirmo-uso-unico --forzar